In [ ]:
import subprocess, sys, importlib.metadata as md, json, os, time
def v(p):
    try: return md.version(p)
    except md.PackageNotFoundError: return "-"
print("before:", {p: v(p) for p in ("torch", "transformers", "compressed-tensors", "accelerate", "safetensors", "huggingface_hub")})
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_tpu==0.2.0", "transformers>=5.5", "compressed-tensors",
                    "--extra-index-url", "https://download.pytorch.org/whl/cpu"], capture_output=True, text=True)
print("pip rc", r.returncode, r.stderr[-1500:])
print("after:", {p: v(p) for p in ("torch", "torch_tpu", "libtpu", "transformers", "compressed-tensors")})

before: {'torch': '2.14.1+cpu', 'transformers': '5.20.1', 'compressed-tensors': '-', 'accelerate': '1.15.0', 'safetensors': '0.8.0', 'huggingface_hub': '1.33.0'}


pip rc 0 
after: {'torch': '2.14.1+cpu', 'torch_tpu': '0.2.0', 'libtpu': '0.0.50', 'transformers': '5.20.1', 'compressed-tensors': '0.19.0'}


In [ ]:
from huggingface_hub import snapshot_download
t = time.time()
path = snapshot_download("xbill9/gemma-4-E2B-it-qat-q4_0-w4a16-ct")
print("download s", round(time.time() - t), path)
print(subprocess.run(f"du -sh {path}/; ls {path}", shell=True, capture_output=True, text=True).stdout)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 50 files:   0%|          | 0/50 [00:00<?, ?it/s]

download s 82 /root/.cache/huggingface/hub/models--xbill9--gemma-4-E2B-it-qat-q4_0-w4a16-ct/snapshots/da64116ff640f56cddb2d4fe20f31e816da22ce9
160K	/root/.cache/huggingface/hub/models--xbill9--gemma-4-E2B-it-qat-q4_0-w4a16-ct/snapshots/da64116ff640f56cddb2d4fe20f31e816da22ce9/
chat_template.jinja
config.json
generation_config.json
model-layer-000.safetensors
model-layer-001.safetensors
model-layer-002.safetensors
model-layer-003.safetensors
model-layer-004.safetensors
model-layer-005.safetensors
model-layer-006.safetensors
model-layer-007.safetensors
model-layer-008.safetensors
model-layer-009.safetensors
model-layer-010.safetensors
model-layer-011.safetensors
model-layer-012.safetensors
model-layer-013.safetensors
model-layer-014.safetensors
model-layer-015.safetensors
model-layer-016.safetensors
model-layer-017.safetensors
model-layer-018.safetensors
model-layer-019.safetensors
model-layer-020.safetensors
model-layer-021.safetensors
model-layer-022.safetensors
model-layer-023.safeten

In [ ]:
SCRIPT = r'''
import sys, time, json, os, torch, traceback
MODEL = "xbill9/gemma-4-E2B-it-qat-q4_0-w4a16-ct"
way = sys.argv[1]
out = {"way": way}
def sync(): torch.accelerator.synchronize()
try:
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(MODEL)
    kw = {"dtype": torch.bfloat16}
    if way != "compressed":
        kw["quantization_config"] = transformers.CompressedTensorsConfig(run_compressed=False)
    t = time.time(); model = AutoModelForCausalLM.from_pretrained(MODEL, **kw).eval(); out["load_cpu_s"] = time.time() - t
    out["model_class"] = type(model).__name__
    out["linear_types"] = sorted({type(m).__name__ for m in model.modules() if "Linear" in type(m).__name__})
    out["param_dtypes"] = sorted({str(p.dtype) for p in model.parameters()})
    t = time.time(); model = model.to("tpu"); sync(); out["to_tpu_s"] = time.time() - t
    msgs = [{"role": "user", "content": "Explain in two sentences why the sky is blue."}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
    ids = {k: v.to("tpu") for k, v in ids.items()}
    n = ids["input_ids"].shape[1]; out["prompt_tokens"] = n
    with torch.no_grad():
        t = time.time(); logits = model(**ids).logits; sync(); out["forward1_s"] = time.time() - t
        out["top_next"] = tok.decode([int(logits[0, -1].argmax().cpu())])
        out["finite"] = bool(torch.isfinite(logits.float()).all().cpu())
        if way in ("eager", "fuse", "compressed"):
            t = time.time(); g = model.generate(**ids, max_new_tokens=8, do_sample=False); sync(); out["gen8_s"] = time.time() - t
            t = time.time(); g = model.generate(**ids, max_new_tokens=32, do_sample=False); sync(); out["gen32_s"] = time.time() - t
            out["text"] = tok.decode(g[0, n:], skip_special_tokens=True)
        elif way == "hfstatic":
            model.forward = torch.compile(model.forward, backend="tpu", dynamic=False)
            t = time.time(); g = model.generate(**ids, max_new_tokens=8, do_sample=False, cache_implementation="static"); sync(); out["gen8_s"] = time.time() - t
            t = time.time(); g = model.generate(**ids, max_new_tokens=32, do_sample=False, cache_implementation="static"); sync(); out["gen32_s"] = time.time() - t
            out["text"] = tok.decode(g[0, n:], skip_special_tokens=True)
        elif way == "buffer":
            SEQ = 256
            def step(tokens, mask, last):
                return model(input_ids=tokens, attention_mask=mask, use_cache=False).logits.index_select(1, last)
            cstep = torch.compile(step, backend="tpu", dynamic=False)
            one = torch.ones((1, 1), dtype=torch.long, device="tpu")
            def run(steps):
                tokens = torch.zeros((1, SEQ), dtype=torch.long); tokens[0, :n] = ids["input_ids"][0].cpu()
                mask = torch.zeros((1, SEQ), dtype=torch.long); mask[0, :n] = 1
                tokens, mask = tokens.to("tpu"), mask.to("tpu"); pos = torch.tensor([n], device="tpu")
                got = []
                for _ in range(steps):
                    nxt = cstep(tokens, mask, pos - 1).argmax(-1)[:, 0:1]
                    got.append(int(nxt.cpu().item()))
                    tokens = tokens.index_copy(1, pos, nxt.view(1, 1)); mask = mask.index_copy(1, pos, one); pos = pos + 1
                return got
            t = time.time(); run(8); out["gen8_s"] = time.time() - t
            t = time.time(); got = run(32); out["gen32_s"] = time.time() - t
            out["text"] = tok.decode(got, skip_special_tokens=True)
    out["hbm"] = {k: getattr(torch.accelerator, k)() for k in ("memory_allocated", "max_memory_allocated") if hasattr(torch.accelerator, k)} if hasattr(torch, "accelerator") else None
except Exception as e:
    out["error"] = f"{type(e).__name__}: {e}"[:1500]; out["tb"] = traceback.format_exc()[-2500:]
print("RESULT " + json.dumps(out, default=str))
'''
open("probe_way.py", "w").write(SCRIPT)
ENV = {"eager": {}, "fuse": {"TORCH_TPU_DEFER_AND_FUSE": "1"}, "buffer": {}, "hfstatic": {}, "compressed": {}}
for way in ("eager", "fuse", "buffer", "hfstatic", "compressed"):
    t = time.time()
    try:
        p = subprocess.run([sys.executable, "probe_way.py", way], env={**os.environ, **ENV[way]}, capture_output=True, text=True, timeout=1500)
        lines = [l for l in p.stdout.splitlines() if l.startswith("RESULT ")]
        print(way, "rc", p.returncode, "wall", round(time.time() - t), "s")
        print(lines[-1][7:] if lines else p.stderr[-3000:])
    except subprocess.TimeoutExpired:
        print(way, "TIMEOUT after 1500 s")

eager rc 0 wall 137 s
{"way": "eager", "load_cpu_s": 14.143022060394287, "model_class": "Gemma4ForConditionalGeneration", "linear_types": ["Gemma4ClippableLinear", "Linear"], "param_dtypes": ["torch.bfloat16", "torch.int64"], "to_tpu_s": 32.195579528808594, "prompt_tokens": 19, "forward1_s": 4.1093339920043945, "top_next": "The", "finite": true, "gen8_s": 6.322345733642578, "gen32_s": 16.98160481452942, "text": "The sky appears blue because of a phenomenon called Rayleigh scattering, where the Earth's atmosphere scatters shorter, blue wavelengths of sunlight more effectively than longer, red", "hbm": {"memory_allocated": 8932352, "max_memory_allocated": 4697647616}}


fuse rc 0 wall 130 s
{"way": "fuse", "load_cpu_s": 29.710089683532715, "model_class": "Gemma4ForConditionalGeneration", "linear_types": ["Gemma4ClippableLinear", "Linear"], "param_dtypes": ["torch.bfloat16", "torch.int64"], "to_tpu_s": 30.404096841812134, "prompt_tokens": 19, "forward1_s": 2.9982075691223145, "top_next": "The", "finite": true, "gen8_s": 13.134002447128296, "gen32_s": 39.943299531936646, "text": "The sky appears blue because of a phenomenon called Rayleigh scattering, where the Earth's atmosphere scatters shorter, blue wavelengths of sunlight more effectively than longer, red", "hbm": {"memory_allocated": 25198080, "max_memory_allocated": 4697647616}}


buffer rc 0 wall 63 s
{"way": "buffer", "load_cpu_s": 13.431765079498291, "model_class": "Gemma4ForConditionalGeneration", "linear_types": ["Gemma4ClippableLinear", "Linear"], "param_dtypes": ["torch.bfloat16", "torch.int64"], "to_tpu_s": 30.51878809928894, "prompt_tokens": 19, "forward1_s": 2.708923101425171, "top_next": "The", "finite": true, "error": "RuntimeError: execute(): failed to prepare compiled mode arguments: failed to get buffer from argument tensor: shape=[1, 256, 8960], strides=[2293760, 8960, 1], numel=2293760, dtype=15, device=cpu, is_contiguous=1, is_cpu=1, is_cuda=0, is_meta=0, storage_offset=0, storage_use_count=2, expected tensor to be on tpu, got cpu", "tb": "tograd/codegen.py:codegen(runtime_wrapper_orchestration)\", line 8, in _runtime_wrapper\n  File \"/usr/local/lib/python3.13/dist-packages/torch/_functorch/_aot_autograd/runtime_wrappers.py\", line 2594, in __call__\n    return self.compiled_fn(*args, **kwargs)\n           ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^\n  

hfstatic rc 0 wall 65 s
{"way": "hfstatic", "load_cpu_s": 13.621440410614014, "model_class": "Gemma4ForConditionalGeneration", "linear_types": ["Gemma4ClippableLinear", "Linear"], "param_dtypes": ["torch.bfloat16", "torch.int64"], "to_tpu_s": 35.41922116279602, "prompt_tokens": 19, "forward1_s": 3.0888662338256836, "top_next": "The", "finite": true, "error": "BackendCompilerFailed: backend='tpu' raised:\nRuntimeError: where(): expected tensor to be on tpu, got cpu\n\nWhile executing %where_self : [num_users=1] = call_function[target=torch.ops.aten.where.self](args = (%bitwise_or_tensor_1, %scalar_tensor_default, %clone_default), kwargs = {})\nOriginal traceback:\n  File \"/usr/local/lib/python3.13/dist-packages/transformers/models/gemma4/modeling_gemma4.py\", line 2312, in forward\n    llm_input_ids = torch.where(multimodal_mask, self.config.text_config.pad_token_id, llm_input_ids)\n\nUse tlparse to see full graph. (https://github.com/pytorch/tlparse?tab=readme-ov-file#tlparse-parse-st

compressed rc 0 wall 56 s
{"way": "compressed", "load_cpu_s": 1.3638498783111572, "model_class": "Gemma4ForConditionalGeneration", "linear_types": ["Gemma4ClippableLinear", "Linear"], "param_dtypes": ["torch.bfloat16", "torch.int32", "torch.int64"], "to_tpu_s": 14.30414080619812, "prompt_tokens": 19, "forward1_s": 15.62313985824585, "top_next": "The", "finite": true, "gen8_s": 5.020793914794922, "gen32_s": 9.205454587936401, "text": "The sky appears blue because of a phenomenon called Rayleigh scattering, where the Earth's atmosphere scatters shorter, blue wavelengths of sunlight more effectively than longer, red", "hbm": {"memory_allocated": 11620571136, "max_memory_allocated": 11758260736}}
